# Laboratorio #8 — Introducción al Deep Learning con PyTorch
Natural Language Processing, Universidad Francisco Marroquín
Autor: George Albadr
Nota: el PDF de instrucciones trae por error el título Laboratorio #7 en el encabezado; formalmente es el Laboratorio #8.

---
### Objetivo
Implementar con PyTorch 2.x los fundamentos prácticos del aprendizaje profundo, en cinco partes:
1. Redes multicapa con nn.Module: resolver XOR, demostrar el colapso lineal y contar los parámetros de una red profunda.
2. Funciones de activación: calcular Sigmoide, Tanh y ReLU y sus derivadas con torch.autograd.
3. Pérdida, optimizador y ciclo de entrenamiento: clasificar Iris con nn.CrossEntropyLoss y comparar tasas de aprendizaje de SGD contra Adam.
4. Redes recurrentes para etiquetado POS: nn.Embedding, luego nn.LSTM o nn.RNN, luego nn.Linear, sobre Penn Treebank con tagset universal y desambiguación de palabras.
5. Conclusiones sobre XOR, representaciones densas y vocabulario desconocido.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import nltk
from nltk.corpus import treebank

# Semilla fija para reproducibilidad requerida en la guía
os.makedirs("figuras", exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Cargar ruta local de datos de NLTK
nltk_local = os.path.abspath('nltk_data')
if os.path.exists(nltk_local) and nltk_local not in nltk.data.path:
    nltk.data.path.insert(0, nltk_local)

# Estilo de gráficas
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print(f"PyTorch versión: {torch.__version__} ejecutando en CPU.")


PyTorch versión: 2.14.1 ejecutando en CPU.


## 1. Redes Multicapa con nn.Module

### 1.1 Resolución de XOR con un perceptrón multicapa
XOR no es linealmente separable; un perceptrón simple no lo resuelve y un MLP con una capa oculta sí. Definimos un MLP con una capa oculta y activación nn.Tanh, lo entrenamos hasta llegar al 100% de exactitud y graficamos su frontera de decisión.

In [ ]:
# Datos XOR
X_xor = torch.tensor([[0.0, 0.0],
                      [0.0, 1.0],
                      [1.0, 0.0],
                      [1.0, 1.0]], dtype=torch.float32)

y_xor = torch.tensor([[0.0],
                      [1.0],
                      [1.0],
                      [0.0]], dtype=torch.float32)

class MLP_XOR(nn.Module):
    def __init__(self, hidden_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden_dim)
        self.act = nn.Tanh()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

torch.manual_seed(42)
mlp_xor = MLP_XOR(hidden_dim=4)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(mlp_xor.parameters(), lr=0.08)

for epoch in range(350):
    optimizer.zero_grad()
    logits = mlp_xor(X_xor)
    loss = criterion(logits, y_xor)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    probs = torch.sigmoid(mlp_xor(X_xor))
    preds = (probs >= 0.5).float()
    acc = (preds == y_xor).float().mean().item()

print(f"Exactitud alcanzada en XOR: {acc * 100:.1f}% (4 de 4 correctos)")
for i in range(4):
    print(f"Entrada: {X_xor[i].tolist()} -> P(y=1): {probs[i].item():.4f} -> Predicción: {int(preds[i].item())} (Real: {int(y_xor[i].item())})")


Exactitud alcanzada en XOR: 100.0% (4 de 4 correctos)
Entrada: [0.0, 0.0] -> P(y=1): 0.0000 -> Predicción: 0 (Real: 0)
Entrada: [0.0, 1.0] -> P(y=1): 0.9978 -> Predicción: 1 (Real: 1)
Entrada: [1.0, 0.0] -> P(y=1): 0.9978 -> Predicción: 1 (Real: 1)
Entrada: [1.0, 1.0] -> P(y=1): 0.0037 -> Predicción: 0 (Real: 0)


In [ ]:
# Graficar frontera de decisión no lineal
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)

with torch.no_grad():
    zz = torch.sigmoid(mlp_xor(grid)).reshape(xx.shape).numpy()

fig, ax = plt.subplots(figsize=(6, 4.5))
cf = ax.contourf(xx, yy, zz, levels=25, cmap='RdBu_r', alpha=0.7)
plt.colorbar(cf, ax=ax, label=r'Probabilidad $P(y=1)$')
ax.contour(xx, yy, zz, levels=[0.5], colors='black', linewidths=2.0, linestyles='--')

for i in range(len(X_xor)):
    clase = int(y_xor[i].item())
    color = '#d62728' if clase == 1 else '#1f77b4'
    marker = 'o' if clase == 1 else 's'
    lbl = f'Clase {clase}' if i < 2 else None
    ax.scatter(X_xor[i, 0], X_xor[i, 1], color=color, s=140, edgecolors='black', 
               linewidths=2, marker=marker, label=lbl, zorder=5)

ax.set_title("Frontera de Decisión No Lineal Aprendida para XOR", fontsize=11, fontweight="bold")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.legend(loc="upper right")
plt.savefig("figuras/01_frontera_xor.png", bbox_inches="tight")
plt.show()


<Figure>

### 1.2 Modelo lineal sin activación y colapso lineal

¿Por qué apilar capas lineales equivale a una sola?
Una capa lineal aplica una matriz y suma un sesgo; la segunda capa hace lo mismo sobre ese resultado. Al combinarlas, el producto de las dos matrices es otra matriz y los sesgos se fusionan en uno solo. Sin una activación no lineal entre ellas, la red entera es una única transformación afín, aunque tenga muchas capas.
Una transformación afín solo traza un hiperplano, que en dos dimensiones es una recta. Por eso esta red no puede resolver XOR.

In [ ]:
class Linear_XOR(nn.Module):
    def __init__(self, hidden_dim=4):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc2(self.fc1(x))

torch.manual_seed(42)
lin_xor = Linear_XOR(hidden_dim=4)
opt_lin = optim.Adam(lin_xor.parameters(), lr=0.08)

for epoch in range(350):
    opt_lin.zero_grad()
    loss = criterion(lin_xor(X_xor), y_xor)
    loss.backward()
    opt_lin.step()

with torch.no_grad():
    preds_lin = (torch.sigmoid(lin_xor(X_xor)) >= 0.5).float()
    acc_lin = (preds_lin == y_xor).float().mean().item()

print(f"Exactitud del modelo lineal sin activación: {acc_lin * 100:.1f}%")
print(f"Predicciones lineales: {preds_lin.ravel().tolist()} vs Reales: {y_xor.ravel().tolist()}")


Exactitud del modelo lineal sin activación: 50.0%
Predicciones lineales: [1.0, 0.0, 1.0, 0.0] vs Reales: [0.0, 1.0, 1.0, 0.0]


### 1.3 Red de la diapositiva, 3 a 4 a 4 a 2, y conteo de parámetros
* Entrada: 3 valores.
* Primera oculta: 4 neuronas, con 12 pesos de la matriz de 4 por 3 más 4 sesgos. Subtotal 16.
* Segunda oculta: 4 neuronas, con 16 pesos de la matriz de 4 por 4 más 4 sesgos. Subtotal 20.
* Salida: 2 neuronas, con 8 pesos de la matriz de 2 por 4 más 2 sesgos. Subtotal 10.
* Total: 16 más 20 más 10 da 46 parámetros.

In [ ]:
class SlideNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 4)
        self.act1 = nn.ReLU()
        self.layer2 = nn.Linear(4, 4)
        self.act2 = nn.ReLU()
        self.layer3 = nn.Linear(4, 2)

    def forward(self, x):
        return self.layer3(self.act2(self.layer2(self.act1(self.layer1(x)))))

red = SlideNetwork()
filas = []
total = 0
for name, p in red.named_parameters():
    filas.append({
        "Parámetro": name,
        "Forma (Shape)": str(list(p.shape)),
        "Tipo": "Pesos" if "weight" in name else "Sesgo",
        "Elementos": p.numel()
    })
    total += p.numel()

df_p = pd.DataFrame(filas)
print(df_p.to_string(index=False))
print(f"\nTotal de parámetros contados con named_parameters(): {total}")


    Parámetro Forma (Shape)  Tipo  Elementos
layer1.weight        [4, 3] Pesos         12
  layer1.bias           [4] Sesgo          4
layer2.weight        [4, 4] Pesos         16
  layer2.bias           [4] Sesgo          4
layer3.weight        [2, 4] Pesos          8
  layer3.bias           [2] Sesgo          2

Total de parámetros contados con named_parameters(): 46


## 2. Funciones de activación y sus derivadas con autograd

Evaluamos Sigmoide, Tanh y ReLU sobre valores de z entre -5 y 5, y usamos torch.autograd.grad para obtener sus derivadas.

In [ ]:
z = torch.linspace(-5.0, 5.0, 200, requires_grad=True)

y_sig = torch.sigmoid(z)
y_tanh = torch.tanh(z)
y_relu = torch.relu(z)

grad_sig = torch.autograd.grad(y_sig.sum(), z, retain_graph=True)[0]
grad_tanh = torch.autograd.grad(y_tanh.sum(), z, retain_graph=True)[0]
grad_relu = torch.autograd.grad(y_relu.sum(), z, retain_graph=False)[0]

z_np = z.detach().numpy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Funciones
ax1.plot(z_np, y_sig.detach().numpy(), label=r"Sigmoide: $\sigma(z)$", color="#1f77b4", lw=2.2)
ax1.plot(z_np, y_tanh.detach().numpy(), label=r"Tanh: $\tanh(z)$", color="#ff7f0e", lw=2.2)
ax1.plot(z_np, y_relu.detach().numpy(), label=r"ReLU: $\max(0, z)$", color="#2ca02c", lw=2.2)
ax1.axhline(0, color="gray", ls="--", lw=0.8)
ax1.set_title("Funciones de Activación", fontsize=11, fontweight="bold")
ax1.set_xlabel("$z$")
ax1.set_ylabel("$f(z)$")
ax1.set_ylim(-1.5, 5.2)
ax1.legend(loc="upper left")

# Derivadas
ax2.plot(z_np, grad_sig.numpy(), label=r"$\sigma'(z)$ [Máx: 0.25]", color="#1f77b4", lw=2.2)
ax2.plot(z_np, grad_tanh.numpy(), label=r"$\tanh'(z)$ [Máx: 1.00]", color="#ff7f0e", lw=2.2)
ax2.plot(z_np, grad_relu.numpy(), label=r"$\mathrm{ReLU}'(z)$ [1 si $z>0$]", color="#2ca02c", lw=2.2)
ax2.axhline(0, color="gray", ls="--", lw=0.8)
ax2.set_title("Derivadas con autograd", fontsize=11, fontweight="bold")
ax2.set_xlabel("$z$")
ax2.set_ylabel("$f'(z)$")
ax2.set_ylim(-0.1, 1.2)
ax2.legend(loc="upper right")

plt.savefig("figuras/02_funciones_activacion_derivadas.png", bbox_inches="tight")
plt.show()


<Figure>

### 2.1 Comentarios sobre las derivadas

¿En qué regiones se desvanece el gradiente?
* Sigmoide: se desvanece cuando el valor absoluto de z supera 2.5, donde la derivada cae por debajo de 0.07, y tiende a cero pasado 4. Su máximo es 0.25 en z igual a 0; cada capa reduce el gradiente al menos a una cuarta parte.
* Tanh: se desvanece cuando el valor absoluto de z supera 2, donde la derivada también cae por debajo de 0.07. Llega a 1 en z igual a 0, pero satura con fuerza en los extremos.

¿Por qué ReLU es la opción por defecto en capas ocultas?
1. Su derivada vale 1 para todo z positivo; el gradiente no se atenúa de forma exponencial al cruzar muchas capas.
2. Calcularla es inmediato, sin funciones trascendentes; solo toma el máximo entre 0 y z.
3. Anula las activaciones con z negativo; la red resultante es dispersa y eso suele ayudar.

## 3. Pérdida, optimizador y ciclo de entrenamiento con Iris

* Partición estratificada: 80% entrenamiento y 20% validación.
* Estandarización: StandardScaler ajustado solo con los datos de entrenamiento.
* Red MLP: 4 entradas, 16 neuronas ocultas con ReLU y 3 salidas, una por especie.
* Pérdida: nn.CrossEntropyLoss.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
target_names = iris.target_names

# División estratificada 80/20
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Estandarización fit solo en train
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_val = scaler.transform(X_val_raw)

X_train_t = torch.tensor(X_tr, dtype=torch.float32)
y_train_t = torch.tensor(y_tr, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

class MLPIris(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(16, 3)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

modelo_iris = MLPIris()
n_params = sum(p.numel() for p in modelo_iris.parameters())
print(f"Muestras Train: {len(X_train_t)} | Val: {len(X_val_t)}")
print(f"Total parámetros de MLPIris (4 -> 16 -> 3): {n_params}")


Muestras Train: 120 | Val: 30
Total parámetros de MLPIris (4 -> 16 -> 3): 131


### 3.1 ¿Por qué la red no debe llevar Softmax al final si usa nn.CrossEntropyLoss?
nn.CrossEntropyLoss ya incluye LogSoftmax y NLLLoss en una sola operación, calculada con el truco Log-Sum-Exp para evitar desbordamientos y valores NaN. Si la red terminara en Softmax, la pérdida recibiría probabilidades y aplicaría un segundo Softmax encima.
Ese doble paso aplana los gradientes y el entrenamiento empeora.

In [ ]:
lrs = [0.001, 0.1, 10.0]
num_epochs = 120
perdidas_sgd = {}

for lr in lrs:
    torch.manual_seed(42)
    m = MLPIris()
    crit = nn.CrossEntropyLoss()
    opt = optim.SGD(m.parameters(), lr=lr)
    
    curva = []
    for ep in range(num_epochs):
        opt.zero_grad()           # 1. Limpiar gradientes
        logits = m(X_train_t)      # 2. Forward
        loss = crit(logits, y_train_t) # 3. Pérdida
        loss.backward()           # 4. Backward
        opt.step()                # 5. Paso optimizador
        curva.append(loss.item())
        
    perdidas_sgd[lr] = curva

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(1, num_epochs + 1), perdidas_sgd[0.001], label=r"SGD $\eta = 0.001$ (Demasiado lento)", color="#1f77b4", lw=2.2)
ax.plot(range(1, num_epochs + 1), perdidas_sgd[0.1], label=r"SGD $\eta = 0.1$ (Convergencia óptima)", color="#2ca02c", lw=2.2)
ax.plot(range(1, num_epochs + 1), perdidas_sgd[10.0], label=r"SGD $\eta = 10.0$ (Divergencia / Oscilación)", color="#d62728", lw=2.2)
ax.set_title("Efecto de la Tasa de Aprendizaje en SGD", fontsize=11, fontweight="bold")
ax.set_xlabel("Época")
ax.set_ylabel("Pérdida (Cross-Entropy)")
ax.set_yscale("log")
ax.legend(loc="upper right")
plt.savefig("figuras/03_curvas_aprendizaje_iris_lr.png", bbox_inches="tight")
plt.show()


<Figure>

### 3.2 Comentarios sobre las tasas de aprendizaje
* Con 0.001 el aprendizaje es lento; tras 120 épocas la pérdida sigue cerca de 1.0.
* Con 0.1 el descenso es rápido y estable; la pérdida termina cerca de 0.20.
* Con 10 los pasos son tan grandes que sobrepasan el valle; la pérdida se dispara por encima de 100 mil y luego oscila cerca de 0.5.
Una tasa pequeña desperdicia épocas y una enorme destruye lo aprendido, así que el punto útil está entre ambas.

In [ ]:
# Comparación con Adam usando su tasa por defecto de 0.001
torch.manual_seed(42)
m_adam = MLPIris()
crit = nn.CrossEntropyLoss()
opt_adam = optim.Adam(m_adam.parameters(), lr=0.001)

perdidas_adam = []
for ep in range(num_epochs):
    opt_adam.zero_grad()
    loss = crit(m_adam(X_train_t), y_train_t)
    loss.backward()
    opt_adam.step()
    perdidas_adam.append(loss.item())

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(range(1, num_epochs + 1), perdidas_sgd[0.1], label=r"Mejor SGD ($\eta = 0.1$)", color="#2ca02c", lw=2.2)
ax.plot(range(1, num_epochs + 1), perdidas_adam, label=r"Adam ($\eta = 0.001$ por defecto)", color="#9467bd", lw=2.2, ls="--")
ax.set_title("Comparación de Convergencia: SGD vs Adam", fontsize=11, fontweight="bold")
ax.set_xlabel("Época")
ax.set_ylabel("Pérdida")
ax.legend(loc="upper right")
plt.savefig("figuras/04_adam_vs_sgd_iris.png", bbox_inches="tight")
plt.show()

# Evaluación en validación con model.eval() y torch.no_grad()
torch.manual_seed(42)
m_final = MLPIris()
opt_f = optim.SGD(m_final.parameters(), lr=0.1)
for ep in range(num_epochs):
    opt_f.zero_grad()
    loss = crit(m_final(X_train_t), y_train_t)
    loss.backward()
    opt_f.step()

m_final.eval()
with torch.no_grad():
    val_logits = m_final(X_val_t)
    val_preds = torch.argmax(val_logits, dim=1)
    val_acc = (val_preds == y_val_t).float().mean().item()

print(f"Exactitud final en validación: {val_acc * 100:.2f}%\n")
print(classification_report(y_val_t.numpy(), val_preds.numpy(), target_names=target_names))

cm = confusion_matrix(y_val_t.numpy(), val_preds.numpy())
fig, ax = plt.subplots(figsize=(4.5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap="Blues", ax=ax, colorbar=False)
ax.grid(False)
ax.set_title("Matriz de Confusión en Validación", fontsize=11, fontweight="bold")
plt.savefig("figuras/05_matriz_confusion_iris.png", bbox_inches="tight")
plt.show()


Exactitud final en validación: 90.00%
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.89      0.80      0.84        10
   virginica       0.82      0.90      0.86        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



<Figure>

<Figure>

### Adam frente a SGD
Adam con su tasa por defecto de 0.001 termina cerca de 0.5; el mejor SGD, con 0.1, termina cerca de 0.20.
Adam adapta el paso por parámetro, pero con una tasa tan baja y solo 120 épocas no alcanza al SGD bien calibrado.

### 3.3 ¿Qué par de especies confunde más el modelo?
* Setosa se clasifica con precisión y recall de 1.00, porque es linealmente separable de las otras dos.
* Versicolor y virginica son el par que se confunde, porque el largo y el ancho de sus pétalos se solapan en la frontera; el recall de versicolor es 0.80 y el de virginica 0.90.
La confusión sale del solapamiento de los datos y no de una falla del modelo.

## 4. Redes recurrentes para etiquetado POS con Penn Treebank

* Corpus: nltk.corpus.treebank.tagged_sents con tagset universal.
* Vocabulario con dos índices especiales: PAD en 0 y UNK en 1.
* Los lotes se arman con nn.utils.rnn.pad_sequence y la pérdida usa ignore_index para ignorar el relleno.
* Arquitectura: nn.Embedding, luego nn.LSTM o nn.RNN, luego nn.Linear.

In [ ]:
tagged_sents = list(treebank.tagged_sents(tagset='universal'))

PAD_TOK = "<PAD>"
UNK_TOK = "<UNK>"

word2idx = {PAD_TOK: 0, UNK_TOK: 1}
tag2idx = {PAD_TOK: 0}

for sent in tagged_sents:
    for word, tag in sent:
        w_low = word.lower()
        if w_low not in word2idx:
            word2idx[w_low] = len(word2idx)
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

idx2tag = {i: t for t, i in tag2idx.items()}

# Partición 80% train / 20% test
split = int(len(tagged_sents) * 0.8)
train_raw, test_raw = tagged_sents[:split], tagged_sents[split:]

print(f"Total oraciones: {len(tagged_sents)} (Train: {len(train_raw)}, Test: {len(test_raw)})")
print(f"Vocabulario palabras: {len(word2idx)} | Etiquetas universales: {len(tag2idx)}")

class TreebankDataset(Dataset):
    def __init__(self, data):
        self.samples = []
        for s in data:
            w_ids = [word2idx.get(w.lower(), word2idx[UNK_TOK]) for w, t in s]
            t_ids = [tag2idx[t] for w, t in s]
            self.samples.append((torch.tensor(w_ids, dtype=torch.long), torch.tensor(t_ids, dtype=torch.long)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]

def collate_fn(batch):
    seqs, tags = zip(*batch)
    p_seqs = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=word2idx[PAD_TOK])
    p_tags = nn.utils.rnn.pad_sequence(tags, batch_first=True, padding_value=tag2idx[PAD_TOK])
    return p_seqs, p_tags

train_loader = DataLoader(TreebankDataset(train_raw), batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(TreebankDataset(test_raw), batch_size=32, shuffle=False, collate_fn=collate_fn)


Total oraciones: 3914 (Train: 3131, Test: 783)
Vocabulario palabras: 11389 | Etiquetas universales: 13


In [ ]:
class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        return self.fc(self.lstm(self.embedding(x))[0])

class RNNTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, nonlinearity='tanh', batch_first=True)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x):
        return self.fc(self.rnn(self.embedding(x))[0])

def train_eval(model, epochs=5):
    crit = nn.CrossEntropyLoss(ignore_index=0)
    opt = optim.Adam(model.parameters(), lr=0.005)
    
    for ep in range(epochs):
        model.train()
        for seqs, tags in train_loader:
            opt.zero_grad()
            logits = model(seqs)
            loss = crit(logits.view(-1, logits.shape[-1]), tags.view(-1))
            loss.backward()
            opt.step()
            
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for seqs, tags in test_loader:
            preds = torch.argmax(model(seqs), dim=-1)
            mask = (tags != 0)
            correct += ((preds == tags) & mask).sum().item()
            total += mask.sum().item()
    return correct / total

torch.manual_seed(42)
tagger_lstm = LSTMTagger(len(word2idx), len(tag2idx))
acc_lstm = train_eval(tagger_lstm, epochs=5)

torch.manual_seed(42)
tagger_rnn = RNNTagger(len(word2idx), len(tag2idx))
acc_rnn = train_eval(tagger_rnn, epochs=5)

df_comp = pd.DataFrame([
    {"Modelo": "nn.LSTM (Long Short-Term Memory)", "Exactitud Test (Sin <PAD>)": f"{acc_lstm * 100:.2f}%"},
    {"Modelo": "nn.RNN (Recurrente Simple con Tanh)", "Exactitud Test (Sin <PAD>)": f"{acc_rnn * 100:.2f}%"}
])
print("--- COMPARACIÓN DE EXACTITUD EN TEST (Penn Treebank) ---")
print(df_comp.to_string(index=False))


--- COMPARACIÓN DE EXACTITUD EN TEST (Penn Treebank) ---
                             Modelo Exactitud Test (Sin <PAD>)
   nn.LSTM (Long Short-Term Memory)                     92.14%
nn.RNN (Recurrente Simple con Tanh)                     92.11%


In [ ]:
def tag_sentence(model, text):
    tokens = text.strip().split()
    ids = [word2idx.get(w.lower(), word2idx[UNK_TOK]) for w in tokens]
    inp = torch.tensor([ids], dtype=torch.long)
    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(inp), dim=-1)[0].numpy()
    return [(w, idx2tag[p]) for w, p in zip(tokens, preds)]

oraciones_test = [
    "I read an interesting book yesterday",
    "I need to book a flight to London",
    "They run fast every morning",
    "She went for a long run in the park"
]

print("--- EVALUACIÓN DE DESAMBIGUACIÓN CONTEXTUAL (LSTM vs RNN) ---\n")
for s in oraciones_test:
    res_lstm = tag_sentence(tagger_lstm, s)
    res_rnn = tag_sentence(tagger_rnn, s)
    anot_lstm = "  ".join([f"{w}/{t}" for w, t in res_lstm])
    anot_rnn  = "  ".join([f"{w}/{t}" for w, t in res_rnn])
    print(f"Oración: '{s}'")
    print(f"  LSTM: {anot_lstm}")
    print(f"  RNN:  {anot_rnn}\n")


--- EVALUACIÓN DE DESAMBIGUACIÓN CONTEXTUAL (LSTM vs RNN) ---

Oración: 'I read an interesting book yesterday'
  LSTM: I/PRON  read/VERB  an/DET  interesting/ADJ  book/NOUN  yesterday/NOUN
  RNN:  I/PRON  read/VERB  an/DET  interesting/ADJ  book/NOUN  yesterday/NOUN

Oración: 'I need to book a flight to London'
  LSTM: I/PRON  need/VERB  to/PRT  book/NOUN  a/DET  flight/NOUN  to/PRT  London/NOUN
  RNN:  I/PRON  need/VERB  to/PRT  book/NOUN  a/DET  flight/NOUN  to/PRT  London/NOUN

Oración: 'They run fast every morning'
  LSTM: They/PRON  run/VERB  fast/ADV  every/DET  morning/NOUN
  RNN:  They/PRON  run/VERB  fast/ADV  every/DET  morning/NOUN

Oración: 'She went for a long run in the park'
  LSTM: She/PRON  went/VERB  for/ADP  a/DET  long/ADJ  run/NOUN  in/ADP  the/DET  park/NOUN
  RNN:  She/PRON  went/VERB  for/ADP  a/DET  long/ADJ  run/NUM  in/ADP  the/DET  park/NOUN



### 4.1 Comentarios sobre ambigüedad y comparación con el Laboratorio 6

¿Resuelve el contexto la ambigüedad? Solo en parte.
* Caso run: la LSTM acierta en los dos usos; la RNN simple falla en uno.
  * En They run fast, el pronombre sujeto lleva a ambos modelos a VERB.
  * En a long run, el artículo y el adjetivo previos llevan a la LSTM a NOUN, pero la RNN simple da NUM. Ahí la memoria de la LSTM sí marca la diferencia.
* Caso book: el contexto izquierdo no alcanza.
  * En an interesting book, ambos modelos dan NOUN, que es lo correcto.
  * En need to book, ambos dan NOUN pese al marcador de infinitivo to, porque en el corpus del Wall Street Journal book casi siempre es un sustantivo comercial. Un modelo bidireccional o uno con atención vería también a flight, que está a la derecha, y lo resolvería.

Comparación con spaCy del Laboratorio 6:
* En el Lab 6, spaCy con en_core_web_sm llegó a cerca de 93.2% en Austen, pero cayó ante cambios de época y de dominio, como Stoker.
* Nuestra LSTM llega a cerca de 92.1% por token en el conjunto de prueba de Penn Treebank.
* spaCy parte de un pipeline preentrenado sobre OntoNotes 5.0, con 1.5 millones de palabras de varios géneros y capas convolucionales con morfología de caracteres. Nuestro modelo aprende los embeddings desde cero con 3,131 oraciones, unas 80 mil palabras, y manda toda palabra no vista a un solo token UNK.
Los dos porcentajes salen de corpus distintos; sirven como orden de magnitud y no como una comparación directa.

## 5. Conclusiones: del perceptrón a la atención

### 5.1 Importancia histórica de XOR e ingrediente clave
En 1969 Minsky y Papert mostraron que un perceptrón de una sola capa no resuelve problemas no lineales, y eso provocó el primer invierno de la IA.
Hicieron falta dos ingredientes juntos. Las capas ocultas llevan los datos a un espacio intermedio; las activaciones no lineales doblan ese espacio. Sin las activaciones, la red colapsa en una sola matriz lineal.

---

### 5.2 One-Hot, BoW y TF-IDF frente a embeddings densos, y el problema OOV
* Las representaciones dispersas, One-Hot, BoW y TF-IDF, son vectores del tamaño del vocabulario donde todas las palabras son ortogonales; no capturan similitud entre sinónimos.
* Los embeddings densos llevan el vocabulario a un espacio continuo de 50 a 300 dimensiones, bajo la hipótesis distribucional de que palabras en contextos parecidos tienen vectores cercanos.
* Con vectores dispersos una palabra nueva no tiene lugar; con embeddings densos y arquitecturas modernas se puede aproximar su posición a partir del contexto.
Los dispersos tratan cada palabra como un símbolo aislado; los densos la ubican entre sus vecinas.

---

### 5.3 Palabras desconocidas en la LSTM y propuestas de mejora
Hoy la red manda toda palabra desconocida a un único token UNK y pierde cualquier rasgo morfológico o sintáctico.
Propuestas de mejora:
1. Tokenización por subpalabras, como BPE o WordPiece: partir palabras en fragmentos frecuentes, por ejemplo unconstitutional en un, constitut e ional, para reconstruir el significado sin caer en UNK.
2. Embeddings por caracteres, como FastText: componer la palabra con n-gramas de caracteres, de modo que reconozca prefijos, sufijos y tiempos verbales.
3. Embeddings contextuales, como Transformers o BERT: generar representaciones que dependen de toda la oración y no de un vector fijo por palabra.